In [2]:
import psycopg2
import pandas as pd
import numpy as np 
import pickle

In [3]:
conn = psycopg2.connect(
    dbname="ncaa",
    host="localhost",
    port=5432
)

cursor = conn.cursor()

In [5]:
players_box = pd.read_csv("../backend/data/players_box.csv", encoding="utf-8")
games = pd.read_csv("../backend/data/games.csv")
teams = pd.read_csv("../backend/data/teams.csv")
conferences = pd.read_csv("../backend/data/conferences.csv")
current_ap = pd.read_csv("../backend/data/current_ap.csv")
matchups = pd.read_csv("../backend/data/matchups.csv")
schedule = pd.read_csv("../backend/data/schedule.csv")
player_photos = pd.read_csv("../backend/data/player_photos.csv")
logo_images = pd.read_csv("../backend/data/logo_images.csv")
team_totals = pd.read_csv("../backend/data/team_totals.csv")

players_box = players_box[players_box.Player != "School Totals"]
players_box['Player'] = players_box['Player'].str.encode('latin1').str.decode('utf8')

player_photos.columns = player_photos.columns.str.strip()
player_photos["Player"] = player_photos["Player"].str.strip()
player_photos["Link"] = player_photos["Link"].str.strip()

schedule["Date"] = pd.to_datetime(schedule["Date"])

In [5]:
player_box_table_cretion_query = """
DROP TABLE IF EXISTS players_box;
CREATE TABLE IF NOT EXISTS players_box (
    gameid VARCHAR,
    player VARCHAR,
    date DATE,
    team VARCHAR,
    opponent VARCHAR,
    venue VARCHAR,
    result VARCHAR,
    role VARCHAR,
    mp INT,
    fgm INT,
    fga INT,
    fg_pct FLOAT,
    fg3m INT,
    fg3a INT,
    fg3_pct FLOAT,
    fg2m INT,
    fg2a INT,
    fg2_pct FLOAT,
    ftm INT,
    fta INT,
    ft_pct FLOAT,
    orb INT,
    drb INT,
    trb INT,
    ast INT,
    stl INT,
    blk INT,
    tov INT,
    pf INT,
    pts INT,
    gmsc FLOAT
);
"""

cursor.execute(player_box_table_cretion_query)
conn.commit()

sql_order = [
    "GameID", "Player", "Date", "Team", "Opponent", "Venue", "Result", "Role",
    "MP", "FG", "FGA", "FG%", "3P", "3PA", "3P%", "2P", "2PA", "2P%",
    "FT", "FTA", "FT%", "ORB", "DRB", "TRB", "AST", "STL", "BLK", "TOV", "PF", "PTS", "GmSc"
]

for index, row in players_box[sql_order].iterrows():
    insert_query = """
    INSERT INTO players_box (
        gameid, player, date, team, opponent, venue, result, role,
        mp, fgm, fga, fg_pct, fg3m, fg3a, fg3_pct, fg2m, fg2a, fg2_pct,
        ftm, fta, ft_pct, orb, drb, trb, ast, stl, blk, tov, pf, pts, gmsc
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    """
    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()   

conn.commit()

In [6]:
games_table_creation_query = """
DROP TABLE IF EXISTS games;

CREATE TABLE IF NOT EXISTS games (
    gameid VARCHAR PRIMARY KEY,
    date DATE,
    venue VARCHAR,
    home_team VARCHAR,
    away_team VARCHAR,
    home_points INT,
    away_points INT,
    home_rank INT,
    away_rank INT,
    winner VARCHAR
);
"""

cursor.execute(games_table_creation_query)
conn.commit()

sql_order_games = [
    "GameID", "Date", "Venue", "Home Team", "Away Team",
    "Home Points", "Away Points", "Home Rank", "Away Rank", "Winner"
]
for index, row in games[sql_order_games].iterrows():
    insert_query = """
    INSERT INTO games (
        gameid, date, venue, home_team, away_team,
        home_points, away_points, home_rank, away_rank, winner
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    """
    row["Home Rank"] = None if row["Home Rank"] == "NR" else int(row["Home Rank"])
    row["Away Rank"] = None if row["Away Rank"] == "NR" else int(row["Away Rank"])

    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()

conn.commit()

In [7]:
teams_table_creation_query = """
DROP TABLE IF EXISTS teams;

CREATE TABLE IF NOT EXISTS teams (
    gameid VARCHAR,
    team VARCHAR,
    location VARCHAR,
    rank INT,
    points INT,
    result VARCHAR
);
"""

cursor.execute(teams_table_creation_query)
conn.commit()

sql_order_teams = [
    "GameID", "Team", "Location", "Rank", "Points", "Result"
]

for index, row in teams[sql_order_teams].iterrows():
    insert_query = """
    INSERT INTO teams (
        gameid, team, location, rank, points, result
    ) VALUES (%s, %s, %s, %s, %s, %s);
    """
    row["Rank"] = None if row["Rank"] == "NR" else int(row["Rank"])
    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()

conn.commit()



In [8]:
conference_table_creation_query = """
DROP TABLE IF EXISTS conferences;
CREATE TABLE IF NOT EXISTS conferences (
    team VARCHAR,
    conference VARCHAR
);
"""

cursor.execute(conference_table_creation_query)
conn.commit()

sql_order_conferences = [
    "Team", "Conference"
]   

for index, row in conferences[sql_order_conferences].iterrows():
    insert_query = """
    INSERT INTO conferences (
        team, conference
    ) VALUES (%s, %s);
    """
    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()

conn.commit()

In [6]:
current_ap_table_creation_query = """
DROP TABLE IF EXISTS current_ap;
CREATE TABLE IF NOT EXISTS current_ap (
    team VARCHAR,
    rank INT
);
"""

cursor.execute(current_ap_table_creation_query)
conn.commit()

sql_order_current_ap = [
    "Team", "rank"
]

for index, row in current_ap[sql_order_current_ap].iterrows():
    insert_query = """
    INSERT INTO current_ap (
        team, rank
    ) VALUES (%s, %s);
    """
    row["rank"] = None if row["rank"] == "NR" else int(row["rank"])
    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()
conn.commit()

In [10]:
matchups_table_creation_query = """
DROP TABLE IF EXISTS matchups;
CREATE TABLE IF NOT EXISTS matchups (
    gameid VARCHAR,
    team VARCHAR,
    opponent VARCHAR,
    date DATE,
    location VARCHAR,
    opponent_location VARCHAR,
    team_poss FLOAT,
    opponent_poss FLOAT,
    team_ortg FLOAT,
    opponent_ortg FLOAT,
    team_drtg FLOAT,
    opponent_drtg FLOAT,
    team_netrtg FLOAT,
    opponent_netrtg FLOAT,
    team_games_before INT,
    opponent_games_before INT,
    team_game_number INT,
    opponent_game_number INT,
    team_wins_before INT,
    team_losses_before INT,
    opponent_wins_before INT,
    opponent_losses_before INT,
    team_winpct_before FLOAT,
    opponent_winpct_before FLOAT,
    team_is_ranked BOOLEAN,
    opponent_is_ranked BOOLEAN,
    team_is_higher_ranked BOOLEAN,
    team_rank INT,
    opponent_rank INT,
    team_win INT,
    team_loss INT,
    opponent_win INT,
    opponent_loss INT,
    team_pts INT,
    opponent_pts INT,
    team_ast INT,
    opponent_ast INT,
    team_trb INT,
    opponent_trb INT,
    team_orb INT,
    opponent_orb INT,
    team_drb INT,
    opponent_drb INT,
    team_stl INT,
    opponent_stl INT,
    team_blk INT,
    opponent_blk INT,
    team_tov INT,
    opponent_tov INT,
    team_pf INT,
    opponent_pf INT,
    team_fgm INT,
    opponent_fgm INT,
    team_fga INT,
    opponent_fga INT,
    team_fg3m INT,
    opponent_fg3m INT,
    team_fg3a INT,
    opponent_fg3a INT,
    team_ftm INT,
    opponent_ftm INT,
    team_fta INT,
    opponent_fta INT,
    team_fg2m INT,
    opponent_fg2m INT,
    team_fg2a INT,
    opponent_fg2a INT
);
"""

cursor.execute(matchups_table_creation_query)
conn.commit()

sql_order_matchups = [
    "GameID", "Team", "opp_Team", "Date", "Location", "opp_Location", "Poss", "opp_Poss",
    "ORtg", "opp_ORtg", "DRtg", "opp_DRtg", "NetRtg", "opp_NetRtg", "games_before", "opp_games_before",
    "team_game_number", "opp_team_game_number", "wins_before", "losses_before", "opp_wins_before", "opp_losses_before",
    "win_pct_before", "opp_win_pct_before", "team_is_ranked", "opp_is_ranked", "team_is_higher_ranked", "Rank", "opp_Rank", 
    "win", "loss", "opp_win", "opp_loss", "PTS", "opp_PTS", "AST", "opp_AST",
    "TRB", "opp_TRB", "ORB", "opp_ORB", "DRB", "opp_DRB",
    "STL", "opp_STL", "BLK", "opp_BLK", "TOV", "opp_TOV", "PF", "opp_PF",
    "FG", "opp_FG", "FGA", "opp_FGA", "3P", "opp_3P", "3PA", "opp_3PA",
    "FT", "opp_FT", "FTA", "opp_FTA", "2P", "opp_2P", "2PA", "opp_2PA"
]

for index, row in matchups[sql_order_matchups].iterrows():
    insert_query = """
    INSERT INTO matchups (
        gameid, team, opponent, date, location, opponent_location, team_poss, opponent_poss,
        team_ortg, opponent_ortg, team_drtg, opponent_drtg, team_netrtg,
        opponent_netrtg, team_games_before, opponent_games_before,
        team_game_number, opponent_game_number, team_wins_before,
        team_losses_before, opponent_wins_before, opponent_losses_before,
        team_winpct_before, opponent_winpct_before, team_is_ranked, opponent_is_ranked, team_is_higher_ranked,
        team_rank, opponent_rank, team_win, team_loss, opponent_win,
        opponent_loss, team_pts, opponent_pts, team_ast, opponent_ast,
        team_trb, opponent_trb, team_orb, opponent_orb, team_drb,
        opponent_drb, team_stl, opponent_stl, team_blk, opponent_blk,
        team_tov, opponent_tov, team_pf, opponent_pf, team_fgm,
        opponent_fgm, team_fga, opponent_fga, team_fg3m,
        opponent_fg3m, team_fg3a, opponent_fg3a, team_ftm,
        opponent_ftm, team_fta, opponent_fta, team_fg2m,
        opponent_fg2m, team_fg2a, opponent_fg2a
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
    );
    """

    row["Rank"] = None if row["Rank"] == "NR" or pd.isna(row["Rank"]) else int(row["Rank"])
    row["opp_Rank"] = None if row["opp_Rank"] == "NR" or pd.isna(row["opp_Rank"]) else int(row["opp_Rank"])
    row["team_is_ranked"] = True if row["team_is_ranked"] == 1 else False
    row["opp_is_ranked"] = True if row["opp_is_ranked"] == 1 else False
    row["team_is_higher_ranked"] = True if row["team_is_higher_ranked"] == 1 else False
    
    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()
    
conn.commit()


In [11]:
schedule_table_creation_query = """
DROP TABLE IF EXISTS schedule;
CREATE TABLE IF NOT EXISTS schedule (
    gameid VARCHAR,
    date DATE,
    team VARCHAR,
    opponent VARCHAR,
    location VARCHAR,
    team_rank INT,
    opponent_rank INT
);
"""

cursor.execute(schedule_table_creation_query)
conn.commit()

sql_order_schedule = [
    "GameID", "Date", "Team", "Opponent", "Location", "Rank", "Opponent Rank"
]

for index, row in schedule[sql_order_schedule].iterrows():
    insert_query = """
    INSERT INTO schedule (
        gameid, date, team, opponent, location, team_rank, opponent_rank
    ) VALUES (%s, %s, %s, %s, %s, %s, %s);
    """
    row["Rank"] = None if row["Rank"] == "NR" else int(row["Rank"])
    row["Opponent Rank"] = None if row["Opponent Rank"] == "NR" else int(row["Opponent Rank"])
    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()

conn.commit()

In [12]:
player_photos_table_creation_query = """
DROP TABLE IF EXISTS player_photos;
CREATE TABLE IF NOT EXISTS player_photos (
    player VARCHAR,
    photo_url VARCHAR
);
"""

cursor.execute(player_photos_table_creation_query)
conn.commit()

sql_order_player_photos = [
    "Player", "Link"
]

for index, row in player_photos[sql_order_player_photos].iterrows():
    insert_query = """
    INSERT INTO player_photos (
        player, photo_url
    ) VALUES (%s, %s);
    """
    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()

conn.commit()

In [13]:
logo_images_table_creation_query = """
DROP TABLE IF EXISTS logo_images;
CREATE TABLE IF NOT EXISTS logo_images (
    team VARCHAR,
    logo_url VARCHAR
);
"""

cursor.execute(logo_images_table_creation_query)
conn.commit()

sql_order_logo_images = [
    "Team", "Link"
]

for index, row in logo_images[sql_order_logo_images].iterrows():
    insert_query = """
    INSERT INTO logo_images (
        team, logo_url
    ) VALUES (%s, %s);
    """
    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()

conn.commit()

In [14]:
team_ratings_data = [{"Team": team_name, **ratings} for team_name, ratings in team_ratings.items()]
team_ratings = pd.DataFrame(team_ratings_data)

In [15]:
team_totals_table_creation_query = """
DROP TABLE IF EXISTS team_totals;
CREATE TABLE IF NOT EXISTS team_totals (
    team VARCHAR,
    mp INT,
    fgm INT,
    fga INT,
    ft INT,
    fta INT,
    fg3m INT,
    fg3a INT,
    orb INT,
    drb INT,
    trb INT,
    ast INT,
    stl INT,
    blk INT,
    tov INT,
    pf INT,
    pts INT,
    wins INT,
    losses INT,
    fg_pct FLOAT,
    fg3_pct FLOAT,
    ft_pct FLOAT,
    fg2_pct FLOAT
);
"""

cursor.execute(team_totals_table_creation_query)
conn.commit()

sql_order_team_totals = [
    "Team", "MP", "FG", "FGA", "FT", "FTA", "3P", "3PA", "ORB", "DRB",
    "TRB", "AST", "STL", "BLK", "TOV", "PF", "PTS", "W", "L",
    "FG%", "3P%", "FT%", "2P%"
]

for index, row in team_totals[sql_order_team_totals].iterrows():
    insert_query = """
    INSERT INTO team_totals (
        team, mp, fgm, fga, ft, fta, fg3m, fg3a, orb, drb,
        trb, ast, stl, blk, tov, pf, pts, wins, losses,
        fg_pct, fg3_pct, ft_pct, fg2_pct
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s
    );
    """
    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()

conn.commit()

In [16]:
team_ratings_table_creation_query = """
DROP TABLE IF EXISTS team_ratings;
CREATE TABLE IF NOT EXISTS team_ratings (
    team VARCHAR,
    adjoff FLOAT,
    adjdef FLOAT,
    adjpythagorean FLOAT
);
"""

cursor.execute(team_ratings_table_creation_query)
conn.commit()

sql_order_team_ratings = [
    "Team", "AdjO", "AdjD", "Pyth"
]

for index, row in team_ratings[sql_order_team_ratings].iterrows():
    insert_query = """
    INSERT INTO team_ratings (
        team, adjoff, adjdef, adjpythagorean
    ) VALUES (%s, %s, %s, %s);
    """
    cursor.execute(insert_query, tuple(row))
    if index % 1000 == 0:
        conn.commit()

conn.commit()